# 01 - Data Preparation and Exploratory Data Analysis

This notebook provides sanity checks and exploratory analysis of the processed KPI Sentinel data.

## Contents
1. Load processed data artifacts
2. Verify row counts and data integrity
3. Check missing price rates
4. Examine example time series
5. Summary statistics by hierarchy level

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.config import DATA_PROCESSED, ITEM_DAILY_FILE, METRIC_FACTS_FILE

## 1. Load Processed Data

In [ ]:
# Load item_daily
item_daily_path = DATA_PROCESSED / ITEM_DAILY_FILE
if item_daily_path.exists():
    item_daily = pd.read_parquet(item_daily_path)
    print(f"Loaded item_daily: {len(item_daily):,} rows")
else:
    print(f"File not found: {item_daily_path}")
    print("Please run: python -m src.m5_ingest")
    item_daily = None

In [ ]:
# Load metric_facts
metric_facts_path = DATA_PROCESSED / METRIC_FACTS_FILE
if metric_facts_path.exists():
    metric_facts = pd.read_parquet(metric_facts_path)
    print(f"Loaded metric_facts: {len(metric_facts):,} rows")
else:
    print(f"File not found: {metric_facts_path}")
    print("Please run: python -m src.kpi_build")
    metric_facts = None

## 2. Data Integrity Checks

In [ ]:
if item_daily is not None:
    print("=== item_daily Summary ===")
    print(f"\nShape: {item_daily.shape}")
    print(f"\nColumns: {list(item_daily.columns)}")
    print(f"\nDate range: {item_daily['date'].min()} to {item_daily['date'].max()}")
    print(f"\nUnique series: {item_daily['series_id'].nunique():,}")
    print(f"Unique stores: {item_daily['store_id'].nunique()}")
    print(f"Unique departments: {item_daily['dept_id'].nunique()}")
    print(f"Unique categories: {item_daily['cat_id'].nunique()}")
    print(f"Unique states: {item_daily['state_id'].nunique()}")

In [ ]:
if metric_facts is not None:
    print("=== metric_facts Summary ===")
    print(f"\nShape: {metric_facts.shape}")
    print(f"\nMetrics: {metric_facts['metric_name'].unique().tolist()}")
    print(f"\nLevels: {metric_facts['level'].unique().tolist()}")
    print(f"\nRecords per metric:")
    print(metric_facts.groupby('metric_name').size())

## 3. Missing Price Analysis

In [ ]:
if item_daily is not None:
    missing_prices = item_daily['sell_price'].isna().sum()
    total_records = len(item_daily)
    missing_pct = (missing_prices / total_records) * 100
    
    print(f"Missing sell_price: {missing_prices:,} ({missing_pct:.2f}%)")
    
    # Missing prices by store
    if missing_prices > 0:
        missing_by_store = item_daily[item_daily['sell_price'].isna()].groupby('store_id').size()
        print(f"\nMissing prices by store (top 5):")
        print(missing_by_store.sort_values(ascending=False).head())

## 4. Example Time Series

In [ ]:
if metric_facts is not None:
    # Plot global revenue over time
    global_revenue = metric_facts[
        (metric_facts['metric_name'] == 'revenue') &
        (metric_facts['level'] == 'global')
    ].sort_values('date')
    
    if len(global_revenue) > 0:
        fig = px.line(
            global_revenue,
            x='date',
            y='value',
            title='Global Daily Revenue'
        )
        fig.update_layout(xaxis_title='Date', yaxis_title='Revenue')
        fig.show()
    else:
        print("No global revenue data available.")

In [ ]:
if metric_facts is not None:
    # Plot units by state
    state_units = metric_facts[
        (metric_facts['metric_name'] == 'units') &
        (metric_facts['level'] == 'state')
    ].sort_values('date')
    
    if len(state_units) > 0:
        fig = px.line(
            state_units,
            x='date',
            y='value',
            color='state_id',
            title='Daily Units by State'
        )
        fig.update_layout(xaxis_title='Date', yaxis_title='Units')
        fig.show()
    else:
        print("No state-level units data available.")

## 5. Summary Statistics

In [ ]:
if metric_facts is not None:
    # Summary stats by metric
    summary = metric_facts.groupby('metric_name')['value'].agg(['mean', 'std', 'min', 'max'])
    print("Summary Statistics by Metric:")
    print(summary.round(2))

In [ ]:
if metric_facts is not None:
    # Check for NaN values in metrics
    nan_counts = metric_facts.groupby('metric_name')['value'].apply(lambda x: x.isna().sum())
    print("\nNaN counts by metric:")
    print(nan_counts)

## Conclusion

This notebook verified:
- Data artifacts are properly loaded
- Row counts and column structures are correct
- Missing price rates are within acceptable bounds
- Time series show expected patterns

Proceed to `02_detection_eval.ipynb` for anomaly detection evaluation.